# TP Integrador — TransAndino Express
**Módulo:** Azure para Ingeniería de Datos — Edición 3  
**Notebook:** `tp_notebook.ipynb`  
**Catálogo Unity:** `smunive_dbw_dataengineered.default`

---
## Índice
1. Parámetro ADF y verificación inicial
2. Capa Bronze
3. Capa Silver
4. Carga incremental con MERGE INTO (incidencias_2024)
5. Capa Gold
6. Celda de limpieza (comentada)

---
## Parte 4.1 — Parámetro de ADF y verificación inicial

In [0]:
# Widget que recibe el parámetro enviado por ADF (Notebook Activity)
# ADF pasa: fecha_proceso = valor de p_fecha_proceso
dbutils.widgets.text("fecha_proceso", "2024-01-15", "Fecha de proceso")
fecha_proceso = dbutils.widgets.get("fecha_proceso")
print(f"Fecha de proceso recibida: {fecha_proceso}")

Fecha de proceso recibida: 2024-01-15


In [0]:
# Verificar acceso al catálogo y al Volume donde llegan los archivos
spark.sql("USE CATALOG smunive_dbw_dataengineered")
spark.sql("USE SCHEMA default")

print("Catálogo activo:", spark.sql("SELECT current_catalog()").collect()[0][0])
print("Schema activo:  ", spark.sql("SELECT current_schema()").collect()[0][0])

# Ruta base en ADLS Gen2 — landing organizado por fecha de proceso
LANDING = "abfss://landing@smunivestad.dfs.core.windows.net"
BRONZE  = "abfss://bronze@smunivestad.dfs.core.windows.net"
SILVER  = "abfss://silver@smunivestad.dfs.core.windows.net"
GOLD    = "abfss://gold@smunivestad.dfs.core.windows.net"

print("Rutas configuradas correctamente.")

Catálogo activo: smunive_dbw_dataengineered
Schema activo:   default
Rutas configuradas correctamente.


---
## Parte 4.2 — Capa Bronze

Reglas:
- Schema explícito con `StructType` — todos los campos `StringType()`
- No se usa `inferSchema=True`
- Se agrega columna `ingestion_timestamp`
- Las tablas se registran en Unity Catalog como `smunive_dbw_dataengineered.default.bronze_*`

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import current_timestamp, lit

# ── Schemas explícitos ─────────────────────────────────────────────────────────

schema_envios = StructType([
    StructField("id_envio",         StringType(), True),
    StructField("id_cliente",       StringType(), True),
    StructField("id_transportista", StringType(), True),
    StructField("id_ruta",          StringType(), True),
    StructField("peso_kg",          StringType(), True),
    StructField("monto_flete",      StringType(), True),
    StructField("fecha_envio",      StringType(), True),
    StructField("estado",           StringType(), True),
    StructField("updated_at",       StringType(), True),
])

schema_transportistas = StructType([
    StructField("id_transportista", StringType(), True),
    StructField("nombre",           StringType(), True),
    StructField("zona",             StringType(), True),
    StructField("ciudad",           StringType(), True),
    StructField("modalidad",        StringType(), True),
])

schema_rutas = StructType([
    StructField("id_ruta",      StringType(), True),
    StructField("origen",       StringType(), True),
    StructField("destino",      StringType(), True),
    StructField("tipo_ruta",    StringType(), True),
    StructField("distancia_km", StringType(), True),
])

schema_clientes = StructType([
    StructField("id_cliente",  StringType(), True),
    StructField("nombre",      StringType(), True),
    StructField("segmento",    StringType(), True),
    StructField("ciudad",      StringType(), True),
    StructField("fecha_alta",  StringType(), True),
])

schema_incidencias = StructType([
    StructField("id_incidencia",     StringType(), True),
    StructField("id_envio",          StringType(), True),
    StructField("tipo_incidencia",   StringType(), True),
    StructField("fecha_incidencia",  StringType(), True),
    StructField("resolucion",        StringType(), True),
    StructField("estado_resolucion", StringType(), True),
])

print("Schemas definidos.")

Schemas definidos.


In [0]:
# ── Función helper para leer CSV y escribir Delta en Bronze ────────────────────

def ingest_to_bronze(filename, schema, table_name):
    """
    Lee un CSV desde landing/<filename>/<fecha_proceso>/<filename>.csv
    Agrega ingestion_timestamp y escribe como Delta en Bronze.
    Registra la tabla en Unity Catalog como smunive_dbw_dataengineered.default.<table_name>.
    """
    path_csv = f"{LANDING}/{filename}/{fecha_proceso}/{filename}.csv"
    path_delta = f"{BRONZE}/{filename}"

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .schema(schema)
        .csv(path_csv)
        .withColumn("ingestion_timestamp", current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(path_delta)
    )

    # Registrar en Unity Catalog
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS smunive_dbw_dataengineered.default.{table_name}
        USING DELTA
        LOCATION '{path_delta}'
    """)

    count = spark.read.format("delta").load(path_delta).count()
    print(f"  {table_name}: {count} filas ingresadas.")


print("Iniciando ingesta Bronze...")
ingest_to_bronze("envios",           schema_envios,         "bronze_envios")
ingest_to_bronze("transportistas",   schema_transportistas, "bronze_transportistas")
ingest_to_bronze("rutas",            schema_rutas,          "bronze_rutas")
ingest_to_bronze("clientes_logistica", schema_clientes,     "bronze_clientes")
ingest_to_bronze("incidencias_2023", schema_incidencias,    "bronze_incidencias")
print("Bronze completo.")

Iniciando ingesta Bronze...
  bronze_envios: 600 filas ingresadas.
  bronze_transportistas: 20 filas ingresadas.
  bronze_rutas: 30 filas ingresadas.
  bronze_clientes: 60 filas ingresadas.
  bronze_incidencias: 120 filas ingresadas.
Bronze completo.


In [0]:
# Verificación rápida — schema de bronze_envios (todos deben ser StringType)
spark.table("smunive_dbw_dataengineered.default.bronze_envios").printSchema()

root
 |-- id_envio: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- id_transportista: string (nullable = true)
 |-- id_ruta: string (nullable = true)
 |-- peso_kg: string (nullable = true)
 |-- monto_flete: string (nullable = true)
 |-- fecha_envio: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



---
## Parte 4.3 — Capa Silver

### silver_envios
- `fecha_envio` → `DateType` resolviendo los 5 formatos
- `peso_kg` y `monto_flete` → `DoubleType`, `N/A` → `null`
- `estado` → minúsculas y sin espacios extra
- Deduplicación por `id_envio`

In [0]:
from pyspark.sql.functions import (
    col, trim, lower, to_date, when, regexp_replace,
    coalesce, year, month, round as spark_round, expr
)
from pyspark.sql.types import DoubleType

df_bronze_envios = spark.table("smunive_dbw_dataengineered.default.bronze_envios")

# ── 1. Normalizar fecha_envio — 5 formatos distintos ──────────────────────────
# Los formatos son: yyyy-MM-dd | MM-dd-yyyy | dd/MM/yyyy | yyyy/MM/dd | yyyy/MM/dd HH:mm:ss

try:
    # Intenta con try_to_date (cluster interactivo con Photon)
    df_silver_envios = df_bronze_envios.withColumn(
        "fecha_envio",
        expr("""
            COALESCE(
                try_to_date(fecha_envio, 'yyyy-MM-dd'),
                try_to_date(fecha_envio, 'MM-dd-yyyy'),
                try_to_date(fecha_envio, 'dd/MM/yyyy'),
                try_to_date(fecha_envio, 'yyyy/MM/dd'),
                try_to_date(fecha_envio, 'yyyy/MM/dd HH:mm:ss')
            )
        """)
    )
    # Fuerza la evaluación para detectar error
    df_silver_envios.limit(1).collect()
except Exception:
    # Fallback con to_date + coalesce
    df_silver_envios = df_bronze_envios.withColumn(
        "fecha_envio",
        coalesce(
            to_date(col("fecha_envio"), "yyyy-MM-dd"),
            to_date(col("fecha_envio"), "MM-dd-yyyy"),
            to_date(col("fecha_envio"), "dd/MM/yyyy"),
            to_date(col("fecha_envio"), "yyyy/MM/dd"),
            to_date(col("fecha_envio"), "yyyy/MM/dd HH:mm:ss")
        )
    )

# ── 2. Convertir peso_kg y monto_flete — N/A → null ──────────────────────────
df_silver_envios = (
    df_silver_envios
    .withColumn(
        "peso_kg",
        when(trim(col("peso_kg")) == "N/A", None)
        .otherwise(col("peso_kg").cast(DoubleType()))
    )
    .withColumn(
        "monto_flete",
        col("monto_flete").cast(DoubleType())
    )
)

# ── 3. Normalizar estado — minúsculas + sin espacios extra ────────────────────
df_silver_envios = df_silver_envios.withColumn(
    "estado",
    trim(lower(col("estado")))
)

# ── 4. Normalizar id_ruta — N/A → null ───────────────────────────────────────
df_silver_envios = df_silver_envios.withColumn(
    "id_ruta",
    when(trim(col("id_ruta")) == "N/A", None).otherwise(col("id_ruta"))
)

# ── 5. Deduplicar por id_envio (quedarse con el registro más reciente) ────────
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_envio = Window.partitionBy("id_envio").orderBy(desc("updated_at"))
df_silver_envios = (
    df_silver_envios
    .withColumn("_rn", row_number().over(window_envio))
    .filter(col("_rn") == 1)
    .drop("_rn", "ingestion_timestamp")
)

# ── Escribir Silver ───────────────────────────────────────────────────────────
path_silver_envios = f"{SILVER}/envios"
(
    df_silver_envios.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(path_silver_envios)
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS smunive_dbw_dataengineered.default.silver_envios
    USING DELTA LOCATION '{path_silver_envios}'
""")

print(f"silver_envios: {df_silver_envios.count()} filas")
df_silver_envios.printSchema()

silver_envios: 600 filas
root
 |-- id_envio: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- id_transportista: string (nullable = true)
 |-- id_ruta: string (nullable = true)
 |-- peso_kg: double (nullable = true)
 |-- monto_flete: double (nullable = true)
 |-- fecha_envio: date (nullable = true)
 |-- estado: string (nullable = true)
 |-- updated_at: string (nullable = true)



In [0]:
# Verificación de calidad silver_envios
from pyspark.sql.functions import count, isnan, isnull

df_check = spark.table("smunive_dbw_dataengineered.default.silver_envios")

print("=== Verificación silver_envios ===")
print("Total filas:", df_check.count())
print("Fechas null (no parseables):", df_check.filter(col("fecha_envio").isNull()).count())
print("peso_kg null (era N/A):",       df_check.filter(col("peso_kg").isNull()).count())
print("id_ruta null (era N/A):",        df_check.filter(col("id_ruta").isNull()).count())
print("Estados únicos:")
df_check.groupBy("estado").count().orderBy("estado").show()

=== Verificación silver_envios ===
Total filas: 600
Fechas null (no parseables): 0
peso_kg null (era N/A): 11
id_ruta null (era N/A): 18
Estados únicos:
+-----------+-----+
|     estado|count|
+-----------+-----+
|  cancelado|   88|
|en_transito|  182|
|  entregado|  190|
|  pendiente|  140|
+-----------+-----+



### Silver — transportistas, rutas, clientes

In [0]:
# ── silver_transportistas ─────────────────────────────────────────────────────
path_silver_tra = f"{SILVER}/transportistas"

df_silver_tra = (
    spark.table("smunive_dbw_dataengineered.default.bronze_transportistas")
    .dropDuplicates(["id_transportista"])
    .drop("ingestion_timestamp")
)

(
    df_silver_tra.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .save(path_silver_tra)
)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS smunive_dbw_dataengineered.default.silver_transportistas
    USING DELTA LOCATION '{path_silver_tra}'
""")
print(f"silver_transportistas: {df_silver_tra.count()} filas")


# ── silver_rutas ──────────────────────────────────────────────────────────────
path_silver_rut = f"{SILVER}/rutas"

df_silver_rut = (
    spark.table("smunive_dbw_dataengineered.default.bronze_rutas")
    .withColumn("distancia_km", col("distancia_km").cast("int"))
    .dropDuplicates(["id_ruta"])
    .drop("ingestion_timestamp")
)

(
    df_silver_rut.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .save(path_silver_rut)
)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS smunive_dbw_dataengineered.default.silver_rutas
    USING DELTA LOCATION '{path_silver_rut}'
""")
print(f"silver_rutas: {df_silver_rut.count()} filas")


# ── silver_clientes ───────────────────────────────────────────────────────────
path_silver_cli = f"{SILVER}/clientes"

df_silver_cli = (
    spark.table("smunive_dbw_dataengineered.default.bronze_clientes")
    .withColumn("fecha_alta", to_date(col("fecha_alta"), "yyyy-MM-dd"))
    .dropDuplicates(["id_cliente"])
    .drop("ingestion_timestamp")
)

(
    df_silver_cli.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .save(path_silver_cli)
)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS smunive_dbw_dataengineered.default.silver_clientes
    USING DELTA LOCATION '{path_silver_cli}'
""")
print(f"silver_clientes: {df_silver_cli.count()} filas")

silver_transportistas: 20 filas
silver_rutas: 30 filas
silver_clientes: 60 filas


### Silver — incidencias (carga inicial con incidencias_2023)

In [0]:
# Carga inicial: solo incidencias_2023
# incidencias_2024 se aplica con MERGE en la Parte 4.4
path_silver_inc = f"{SILVER}/incidencias"

df_silver_inc = (
    spark.table("smunive_dbw_dataengineered.default.bronze_incidencias")
    .withColumn("fecha_incidencia", to_date(col("fecha_incidencia"), "yyyy-MM-dd"))
    .dropDuplicates(["id_incidencia"])
    .drop("ingestion_timestamp")
)

(
    df_silver_inc.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .save(path_silver_inc)
)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS smunive_dbw_dataengineered.default.silver_incidencias
    USING DELTA LOCATION '{path_silver_inc}'
""")
print(f"silver_incidencias (carga inicial 2023): {df_silver_inc.count()} filas")

silver_incidencias (carga inicial 2023): 120 filas


---
## Parte 4.4 — Carga incremental con MERGE INTO

`incidencias_2024.csv` contiene dos tipos de registros mezclados:
- 30 actualizaciones a registros existentes de 2023 (`estado_resolucion` → `"Resuelto"`)
- 50 nuevas incidencias de 2024

Se usa `MERGE INTO` con condición de match por `id_incidencia`.

In [0]:
# Leer incidencias_2024 desde landing
path_inc_2024_csv = f"{LANDING}/incidencias_2024/{fecha_proceso}/incidencias_2024.csv"

df_inc_2024 = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .schema(schema_incidencias)
    .csv(path_inc_2024_csv)
    .withColumn("fecha_incidencia", to_date(col("fecha_incidencia"), "yyyy-MM-dd"))
    .dropDuplicates(["id_incidencia"])
)

print(f"Registros en incidencias_2024: {df_inc_2024.count()}")

# Crear vista temporal para usarla en el MERGE
df_inc_2024.createOrReplaceTempView("incidencias_2024_staged")

Registros en incidencias_2024: 80


In [0]:
# MERGE INTO silver_incidencias
# - WHEN MATCHED: actualiza estado_resolucion y resolucion
# - WHEN NOT MATCHED: inserta la fila completa

spark.sql(f"""
    MERGE INTO delta.`{path_silver_inc}` AS target
    USING incidencias_2024_staged AS source
    ON target.id_incidencia = source.id_incidencia

    WHEN MATCHED THEN UPDATE SET
        target.estado_resolucion = source.estado_resolucion,
        target.resolucion        = source.resolucion

    WHEN NOT MATCHED THEN INSERT (
        id_incidencia,
        id_envio,
        tipo_incidencia,
        fecha_incidencia,
        resolucion,
        estado_resolucion
    ) VALUES (
        source.id_incidencia,
        source.id_envio,
        source.tipo_incidencia,
        source.fecha_incidencia,
        source.resolucion,
        source.estado_resolucion
    )
""")

total_post_merge = spark.read.format("delta").load(path_silver_inc).count()
print(f"silver_incidencias post-MERGE: {total_post_merge} filas")
print("Esperado: 120 (2023) + 50 (nuevas 2024) = 170 filas")

silver_incidencias post-MERGE: 170 filas
Esperado: 120 (2023) + 50 (nuevas 2024) = 170 filas


In [0]:
# Delta Time Travel — verificar que la versión anterior (v0) sigue accesible
df_version_0 = spark.read.format("delta").option("versionAsOf", 0).load(path_silver_inc)
df_version_1 = spark.read.format("delta").option("versionAsOf", 1).load(path_silver_inc)

print(f"Versión 0 (carga inicial 2023):  {df_version_0.count()} filas")
print(f"Versión 1 (post-MERGE con 2024): {df_version_1.count()} filas")

# Ver historial completo de la tabla
spark.sql(f"DESCRIBE HISTORY delta.`{path_silver_inc}`").select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

Versión 0 (carga inicial 2023):  120 filas
Versión 1 (post-MERGE con 2024): 170 filas
+-------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationParameters                                                                                                                                                                                                                         |
+-------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2      |2026-06-25 00:46:07|OPTIMIZE |{clusterBy -> [], zOrderBy -> [], batchId ->

---
## Parte 4.5 — Capa Gold

### gold_envios_zona
Join `silver_envios` ↔ `silver_transportistas` → zona  
Agrupado por zona, año y mes  
KPIs: total envíos, monto total, peso total, ticket promedio

In [0]:
from pyspark.sql.functions import sum as spark_sum, count as spark_count, avg

df_envios = spark.table("smunive_dbw_dataengineered.default.silver_envios")
df_tra    = spark.table("smunive_dbw_dataengineered.default.silver_transportistas")
df_rutas  = spark.table("smunive_dbw_dataengineered.default.silver_rutas")

# ── gold_envios_zona ──────────────────────────────────────────────────────────
df_gold_zona = (
    df_envios
    .join(df_tra.select("id_transportista", "zona"), on="id_transportista", how="left")
    .withColumn("anio", year(col("fecha_envio")))
    .withColumn("mes",  month(col("fecha_envio")))
    .groupBy("zona", "anio", "mes")
    .agg(
        spark_count("id_envio")         .alias("total_envios"),
        spark_sum("monto_flete")         .alias("monto_total"),
        spark_sum("peso_kg")             .alias("peso_total"),
    )
    .withColumn(
        "ticket_promedio",
        spark_round(col("monto_total") / col("total_envios"), 2)
    )
    .withColumn("monto_total",  spark_round(col("monto_total"),  2))
    .withColumn("peso_total",   spark_round(col("peso_total"),   2))
    .orderBy("zona", "anio", "mes")
)

path_gold_zona = f"{GOLD}/envios_zona"
(
    df_gold_zona.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .save(path_gold_zona)
)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS smunive_dbw_dataengineered.default.gold_envios_zona
    USING DELTA LOCATION '{path_gold_zona}'
""")
print(f"gold_envios_zona: {df_gold_zona.count()} filas")
df_gold_zona.show(10)

gold_envios_zona: 151 filas
+------+----+---+------------+-----------+----------+---------------+
|  zona|anio|mes|total_envios|monto_total|peso_total|ticket_promedio|
+------+----+---+------------+-----------+----------+---------------+
|Centro|2022|  6|           3|   10804.99|    839.26|        3601.66|
|Centro|2022|  7|           3|    8296.54|   1657.68|        2765.51|
|Centro|2022|  8|           3|   13337.82|   1598.38|        4445.94|
|Centro|2022|  9|           4|   10753.17|   2011.13|        2688.29|
|Centro|2022| 10|           2|    5127.27|    747.75|        2563.64|
|Centro|2022| 11|           4|   12576.91|   1592.61|        3144.23|
|Centro|2022| 12|           2|    7537.29|   1375.25|        3768.65|
|Centro|2023|  1|           6|   27304.91|    2199.2|        4550.82|
|Centro|2023|  2|           3|    13469.3|   1325.86|        4489.77|
|Centro|2023|  3|           6|   27870.92|   3152.33|        4645.15|
+------+----+---+------------+-----------+----------+---------

### gold_envios_tipo_ruta
Join `silver_envios` ↔ `silver_rutas` → tipo_ruta  
Agrupado por tipo de ruta, año y mes  
KPIs: total envíos, monto total, ticket promedio, distancia promedio

In [0]:
# ── gold_envios_tipo_ruta ─────────────────────────────────────────────────────
df_gold_ruta = (
    df_envios
    .join(
        df_rutas.select("id_ruta", "tipo_ruta", "distancia_km"),
        on="id_ruta", how="left"
    )
    .withColumn("anio", year(col("fecha_envio")))
    .withColumn("mes",  month(col("fecha_envio")))
    .groupBy("tipo_ruta", "anio", "mes")
    .agg(
        spark_count("id_envio")              .alias("total_envios"),
        spark_sum("monto_flete")              .alias("monto_total"),
        avg(col("distancia_km").cast("double")).alias("distancia_promedio"),
    )
    .withColumn(
        "ticket_promedio",
        spark_round(col("monto_total") / col("total_envios"), 2)
    )
    .withColumn("monto_total",         spark_round(col("monto_total"),         2))
    .withColumn("distancia_promedio",  spark_round(col("distancia_promedio"),  2))
    .orderBy("tipo_ruta", "anio", "mes")
)

path_gold_ruta = f"{GOLD}/envios_tipo_ruta"
(
    df_gold_ruta.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .save(path_gold_ruta)
)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS smunive_dbw_dataengineered.default.gold_envios_tipo_ruta
    USING DELTA LOCATION '{path_gold_ruta}'
""")
print(f"gold_envios_tipo_ruta: {df_gold_ruta.count()} filas")
df_gold_ruta.show(10)

gold_envios_tipo_ruta: 108 filas
+---------+----+---+------------+-----------+------------------+---------------+
|tipo_ruta|anio|mes|total_envios|monto_total|distancia_promedio|ticket_promedio|
+---------+----+---+------------+-----------+------------------+---------------+
|     NULL|2022|  7|           1|    2678.73|              NULL|        2678.73|
|     NULL|2022|  9|           1|    4875.65|              NULL|        4875.65|
|     NULL|2022| 10|           1|     7801.8|              NULL|         7801.8|
|     NULL|2022| 11|           2|   11364.22|              NULL|        5682.11|
|     NULL|2023|  3|           1|    1354.36|              NULL|        1354.36|
|     NULL|2023|  5|           1|    5226.93|              NULL|        5226.93|
|     NULL|2023|  6|           1|    1874.22|              NULL|        1874.22|
|     NULL|2023|  7|           1|     132.03|              NULL|         132.03|
|     NULL|2023|  8|           2|    9925.23|              NULL|        4962

In [0]:
# ── Verificación cruzada Gold vs Silver ───────────────────────────────────────
# La suma de total_envios en gold_envios_zona debe ser ≈ total filas silver_envios
# (tolerancia de 10 filas para envíos sin transportista)

total_silver  = spark.table("smunive_dbw_dataengineered.default.silver_envios").count()
total_gold    = (
    spark.table("smunive_dbw_dataengineered.default.gold_envios_zona")
    .agg(spark_sum("total_envios").alias("suma"))
    .collect()[0]["suma"]
)
diferencia = abs(total_silver - total_gold)

print(f"Total filas silver_envios:           {total_silver}")
print(f"Suma total_envios en gold_zona:      {total_gold}")
print(f"Diferencia:                          {diferencia}")
print(f"Validación (tolerancia 10):          {'OK' if diferencia <= 10 else 'REVISAR'}")

Total filas silver_envios:           600
Suma total_envios en gold_zona:      600
Diferencia:                          0
Validación (tolerancia 10):          OK


---
## Parte 4.6 — Celda de limpieza

Todos los comandos están comentados. Descomentar solo si se necesita reiniciar el entorno.

In [0]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELDA DE LIMPIEZA — COMENTADA POR DEFECTO                      ║
# ║  Descomentar y ejecutar solo para resetear el entorno           ║
# ╚══════════════════════════════════════════════════════════════════╝

# TABLAS BRONZE
#spark.sql("DROP TABLE IF EXISTS smunive_dbw_dataengineered.default.bronze_envios")
#spark.sql("DROP TABLE IF EXISTS smunive_dbw_dataengineered.default.bronze_transportistas")
#spark.sql("DROP TABLE IF EXISTS smunive_dbw_dataengineered.default.bronze_rutas")
#spark.sql("DROP TABLE IF EXISTS smunive_dbw_dataengineered.default.bronze_clientes")
#spark.sql("DROP TABLE IF EXISTS smunive_dbw_dataengineered.default.bronze_incidencias")

# TABLAS SILVER
#spark.sql("DROP TABLE IF EXISTS smunive_dbw_dataengineered.default.silver_envios")
#spark.sql("DROP TABLE IF EXISTS smunive_dbw_dataengineered.default.silver_transportistas")
#spark.sql("DROP TABLE IF EXISTS smunive_dbw_dataengineered.default.silver_rutas")
#spark.sql("DROP TABLE IF EXISTS smunive_dbw_dataengineered.default.silver_clientes")
#spark.sql("DROP TABLE IF EXISTS smunive_dbw_dataengineered.default.silver_incidencias")

# TABLAS GOLD
#spark.sql("DROP TABLE IF EXISTS smunive_dbw_dataengineered.default.gold_envios_zona")
#spark.sql("DROP TABLE IF EXISTS smunive_dbw_dataengineered.default.gold_envios_tipo_ruta")

# ARCHIVOS DELTA EN ADLS (elimina los datos físicos)
#dbutils.fs.rm(f"{BRONZE}", recurse=True)
#dbutils.fs.rm(f"{SILVER}", recurse=True)
#dbutils.fs.rm(f"{GOLD}",   recurse=True)

print("Celda de limpieza lista (todo comentado).")

Celda de limpieza lista (todo comentado).
